# Exercise 9: Chromaticity

**Block 3: Dispersion and chromaticity**

**Learning objectives**
* Compute the tune as a function of momentum, $Q(\delta)$, and fit the chromaticity $Q' = \text{d}Q/\text{d}\delta$.
* Compare with the formula for the **natural chromaticity** $Q' = −(1/4\pi) \oint \beta(s) k(s) \text{d}s$.
* See chromaticity in phase space: particles with different momenta drift apart in phase.

**Motivation.** A particle with more momentum is stiffer: every quadrupole focuses it a bit less,
like a lens with a longer focal length. Less focusing means a lower tune. The amount by which
the tune changes with momentum is the chromaticity (the name comes from optics, where lenses
focus different colours differently).

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from beamlib import elements as E, optics as O, lattices as L, tracking as T, diagnostics as D, plotting as P
from beamlib.plotting import COLORS, MM
P.setup()

ring = L.fodo_ring()

## 1. Tune vs momentum

In `beamlib` a quadrupole of strength $k$ acts on a particle with momentum deviation $\delta$ with strength
$k/(1 + \delta)$. `O.one_turn_matrix(ring, delta)` builds the one-turn matrix for that $\delta$.
(Our simple model keeps the dipoles' weak focusing independent of $\delta$.)

**Task.** Implement `tunes_vs_delta`, returning the fractional tunes $Q_x(\delta)$, $Q_y(\delta)$. Then fit a
straight line to find $Q_x'$ and $Q_y'$.

In [ ]:
def tunes_vs_delta(ring, deltas):
    """Arrays Qx(delta), Qy(delta) (fractional tunes) from the one-turn matrix."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
deltas = np.linspace(-5e-3, 5e-3, 21)
Qx, Qy = tunes_vs_delta(ring, deltas)
# YOUR CODE HERE
raise NotImplementedError("Replace this line with your code")
print(f"Q'x = {chroma_x:.4f},  Q'y = {chroma_y:.4f}")
assert chroma_x < 0 and chroma_y < 0, "natural chromaticity is negative"
assert np.isclose(chroma_x, -1.56, atol=0.01) and np.isclose(chroma_y, -1.97, atol=0.01)
print("GOOD! Chromaticity fitted")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(deltas * 1e3, Qx, "o", color=COLORS["x"], label=rf"$Q_x$ (fractional), $Q_x'$ = {chroma_x:.2f}")
ax.plot(deltas * 1e3, Qy - 0.6, "s", color=COLORS["y"], label=rf"$Q_y$ (fractional) - 0.6, $Q_y'$ = {chroma_y:.2f}")
ax.set_xlabel(r"$\delta$ [$10^{-3}$]"); ax.set_ylabel("tune"); ax.legend()
plt.show()

## 2. The natural chromaticity formula

A thin quadrupole error $\Delta kl$ at a place with beta function $\beta$ shifts the tune by $\Delta Q = \beta \Delta kl/4\pi$. 
For a particle with momentum deviation $\delta$ every quad changes by $\Delta k \approx -k\delta$, so

$$Q' = -\frac{1}{4\pi}\oint \beta(s)\,k(s)\,ds$$

where $k$ is the focusing strength in that plane (in $y$ the signs of all quads flip).

**Task.** Evaluate the integral numerically. `O.ring_twiss(ring, n_slices=n)` returns $\beta$ at the end of
every slice (`tw["betx"]`), and `tw["elem"]` tells you which element each slice belongs to. For each
quadrupole slice of length $L/n$, use the average of $\beta$ at the start and end of the slice.

In [ ]:
def natural_chromaticity(ring, n_slices: int = 100):
    """(Q'x, Q'y) from -1/(4 pi) * integral(beta * k ds) over the quadrupoles."""
    tw = O.ring_twiss(ring, n_slices=n_slices)
    xi_x = xi_y = 0.0
    for i, el in enumerate(ring):
        if not isinstance(el, E.Quadrupole):
            continue
        # YOUR CODE HERE
        raise NotImplementedError("Replace this line with your code")
    return xi_x, xi_y

In [ ]:
xi_x, xi_y = natural_chromaticity(ring)
print(f"formula: Q'x = {xi_x:.5f}, Q'y = {xi_y:.5f}")
h = 1e-6
dq = (np.array(O.tunes_from_matrix(O.one_turn_matrix(ring, h)))
      - np.array(O.tunes_from_matrix(O.one_turn_matrix(ring, -h)))) / (2 * h)
print(f"tracking (exact derivative): Q'x = {dq[0]:.5f}, Q'y = {dq[1]:.5f}")
assert np.allclose([xi_x, xi_y], dq, rtol=1e-3)
print("GOOD! The natural chromaticity formula agrees with the one-turn matrix")

The fit in part 1 is slightly different from the exact derivative because $Q(\delta)$ is not perfectly
straight (there is a small $\delta^2$ term). The formula gives the slope at $\delta$ = 0.

## 3. Chromaticity in phase space

**Predict first.** Two particles with $\delta = \pm2\cdot10^{-3}$ start at the same phase. In normalised phase space
(each one around its *own* closed orbit), how will the angle between them evolve?

*Your prediction:*

In [ ]:
one_turn = T.make_one_turn_map(ring)
twx = O.twiss_from_matrix(O.one_turn_matrix(ring)[:2, :2])
dls = np.array([-2e-3, 2e-3])
# closed orbits as columns: shape (4, 2), one column per particle
cos = np.column_stack([O.closed_orbit_linear(O.one_turn_matrix(ring, d), d) for d in dls])
bunch = np.vstack([cos, dls])               # shape (5, 2)
bunch[0] += 2e-3                            # same betatron amplitude and phase
res = T.track(bunch, one_turn, 200)
# betatron part = coordinates minus the closed orbit
xb, pb = res.x - cos[0], res.px - cos[1]
_, phi = D.action_angle(xb, pb, twx["beta"], twx["alpha"])
dphi = np.unwrap(phi[:, 0] - phi[:, 1])

Q2 = np.array([O.tunes_from_matrix(O.one_turn_matrix(ring, d))[0] for d in dls])
Q0 = O.tunes_from_matrix(O.one_turn_matrix(ring))[0]
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
for k, col in enumerate([COLORS["y"], COLORS["x"]]):
    J, ph_k = D.action_angle(xb[:, k], pb[:, k], twx["beta"], twx["alpha"])
    rel = np.unwrap(ph_k) - 2 * np.pi * Q0 * np.arange(len(ph_k))   # phase seen from a frame
    r = np.sqrt(2 * J)                                               # rotating with the on-momentum tune
    sc = axs[0].scatter(r * np.cos(rel) * MM, -r * np.sin(rel) * MM, c=np.arange(len(rel)), s=8,
                        cmap="viridis", marker="o" if k == 0 else "s")
    axs[0].annotate(rf"$\delta$ = {dls[k]:+.0e}", (r[-1] * np.cos(rel[-1]) * MM, -r[-1] * np.sin(rel[-1]) * MM),
                    color=col, xytext=(5, 5), textcoords="offset points")
plt.colorbar(sc, ax=axs[0], label="turn")
P.phase_space_labels(axs[0], "x", True); axs[0].set_aspect("equal")
axs[0].set_title("seen from a frame rotating with the on-momentum tune")
axs[1].plot(np.degrees(dphi), color=COLORS["accent"], label="measured")
axs[1].plot(np.degrees(2 * np.pi * (Q2[0] - Q2[1]) * np.arange(len(dphi))), "k--", label=r"$2\pi\,Q'\,\Delta\delta\,n$")
axs[1].set_xlabel(r"turn $n$"); axs[1].set_ylabel("phase difference [deg]"); axs[1].legend()
plt.show()

The phase difference grows linearly: after $n$ turns it is $2\pi Q' \Delta \delta n$. A beam with a momentum
spread therefore has a **tune spread** $\Delta Q \approx |Q'| \sigma_\delta$; this matters for resonances (Block 4)
and for Landau damping (Block 5).

## Reflection questions

1. Why is the natural chromaticity always negative?
2. Our ring has $Q' \approx$ −1.6 and −2.0. The LHC's natural chromaticity is about −100. What makes it
   so much larger? (Look at the formula.)
3. With $\sigma_\delta = 10^{-3}$, what is the tune spread in our ring? Is that a lot?